# Rollenmodell und Clustering-Vergleich

Dieses Notebook bewertet die transparente Multi-Label-Rollenbaseline und vergleicht sie mit einem unüberwachten K-Means-Clustering. Ziel ist nicht, die Regeln nachträglich durch Cluster zu rechtfertigen, sondern zu prüfen, ob die sechs relativen Basiswertanteile stabile und fachlich interpretierbare Gruppen bilden.

Die Cluster werden ausschließlich aus relativen Werteprofilen gebildet. Absolute Stärke, Generation, Evolutionsstufe und die regelbasierten Rollen fließen nicht in das Training ein.

In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

from pokemon_team_advisor.roles import Role, analyze_pokemon_roles

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "pokemon.csv"
FIGURE_DIRECTORY = PROJECT_ROOT / "reports" / "figures"

STAT_COLUMNS = [
    "hp",
    "attack",
    "defense",
    "special_attack",
    "special_defense",
    "speed",
]
SEEDS = [7, 23, 47, 71, 101]
SELECTED_CLUSTER_COUNT = 4
RANDOM_STATE = 7

## Datengrundlage

Die Analyse verwendet denselben validierten Snapshot wie EDA, SQL-Analysen und Anwendung.

In [ ]:
data = pd.read_csv(DATA_PATH)

required_columns = {
    "id",
    "name",
    "base_stat_total",
    "evolution_stage",
    "is_final_evolution",
    *STAT_COLUMNS,
}
missing_columns = sorted(required_columns.difference(data.columns))
if missing_columns:
    raise ValueError(f"Missing columns: {', '.join(missing_columns)}")

if data["id"].duplicated().any():
    raise ValueError("Pokemon ids must be unique.")

calculated_totals = data[STAT_COLUMNS].sum(axis=1)
if not calculated_totals.equals(data["base_stat_total"]):
    raise ValueError("At least one base_stat_total is inconsistent.")

print(f"Rows: {len(data):,}")
print(f"Distinct Pokemon: {data['id'].nunique():,}")
data.head()

## Regelbasierte Multi-Label-Baseline

`fit_score` beschreibt die relative Profilpassung. `strength_score` bewertet die absoluten, für die Rolle relevanten Werte innerhalb der Referenzmenge. Ein Pokémon kann bis zu drei Rollen erhalten.

In [ ]:
role_columns = ["id", "name", *STAT_COLUMNS, "base_stat_total"]
role_analysis = analyze_pokemon_roles(
    data[role_columns].to_dict(orient="records")
)

assigned_roles = pd.DataFrame(
    [
        {
            "id": pokemon_id,
            "role": score["role"].value,
            "fit_score": score["fit_score"],
            "strength_score": score["strength_score"],
        }
        for pokemon_id, scores in role_analysis.items()
        for score in scores
        if score["assigned"]
    ]
)

role_summary = (
    assigned_roles.groupby("role", as_index=False)
    .agg(
        pokemon_count=("id", "count"),
        average_fit=("fit_score", "mean"),
        average_strength=("strength_score", "mean"),
    )
    .sort_values("pokemon_count", ascending=False)
)
role_summary["pokemon_percentage"] = (
    role_summary["pokemon_count"] / len(data) * 100
).round(1)
role_summary[["average_fit", "average_strength"]] = role_summary[
    ["average_fit", "average_strength"]
].round(3)

roles_per_pokemon = assigned_roles.groupby("id").size().value_counts().sort_index()
display(role_summary)
display(roles_per_pokemon.rename("pokemon_count").to_frame())

## Relative und skalierte Werteprofile

Jeder Basiswert wird durch den Gesamtbasiswert desselben Pokémon geteilt. Dadurch beschreibt jede Zeile eine Zusammensetzung mit Summe eins. `StandardScaler` verhindert anschließend, dass Merkmale mit größerer Streuung die euklidische Distanz dominieren.

In [ ]:
relative_stats = data[STAT_COLUMNS].div(data["base_stat_total"], axis=0)
if not np.allclose(relative_stats.sum(axis=1), 1.0):
    raise ValueError("Relative stat profiles must sum to one.")

scaler = StandardScaler()
scaled_stats = scaler.fit_transform(relative_stats)

pd.DataFrame(
    {
        "scaled_mean": scaled_stats.mean(axis=0),
        "scaled_std": scaled_stats.std(axis=0),
    },
    index=STAT_COLUMNS,
).round(6)

## Clusterzahl und Stabilität

Für `k=2` bis `k=10` werden fünf Zufallsstarts mit jeweils `n_init=20` verglichen. Silhouette und Davies-Bouldin bewerten die Trennung; der mittlere paarweise Adjusted Rand Index bewertet die Stabilität gegenüber dem Zufallsstart.

In [ ]:
evaluation_rows = []

for cluster_count in range(2, 11):
    label_runs = []
    silhouette_values = []
    davies_bouldin_values = []
    cluster_sizes = []

    for seed in SEEDS:
        candidate = KMeans(
            n_clusters=cluster_count,
            random_state=seed,
            n_init=20,
        )
        labels = candidate.fit_predict(scaled_stats)
        label_runs.append(labels)
        silhouette_values.append(silhouette_score(scaled_stats, labels))
        davies_bouldin_values.append(
            davies_bouldin_score(scaled_stats, labels)
        )
        cluster_sizes.append(np.bincount(labels, minlength=cluster_count))

    stability_values = [
        adjusted_rand_score(label_runs[left], label_runs[right])
        for left, right in combinations(range(len(label_runs)), 2)
    ]

    evaluation_rows.append(
        {
            "clusters": cluster_count,
            "silhouette_mean": np.mean(silhouette_values),
            "silhouette_std": np.std(silhouette_values),
            "davies_bouldin_mean": np.mean(davies_bouldin_values),
            "stability_ari": np.mean(stability_values),
            "smallest_cluster": min(int(sizes.min()) for sizes in cluster_sizes),
            "largest_cluster": max(int(sizes.max()) for sizes in cluster_sizes),
        }
    )

cluster_metrics = pd.DataFrame(evaluation_rows)
display(cluster_metrics.round(3))

In [ ]:
FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metric_specs = [
    ("silhouette_mean", "Silhouette (höher ist besser)", (0.0, 0.22)),
    ("davies_bouldin_mean", "Davies-Bouldin (niedriger ist besser)", (0.0, 2.0)),
    ("stability_ari", "Stabilität: ARI (höher ist besser)", (0.0, 1.05)),
]

for axis, (column, title, limits) in zip(axes, metric_specs, strict=True):
    axis.plot(
        cluster_metrics["clusters"],
        cluster_metrics[column],
        marker="o",
    )
    selected_row = cluster_metrics.loc[
        cluster_metrics["clusters"] == SELECTED_CLUSTER_COUNT
    ].iloc[0]
    axis.scatter(
        [SELECTED_CLUSTER_COUNT],
        [selected_row[column]],
        s=90,
        label="gewähltes k=4",
        zorder=3,
    )
    axis.set(title=title, xlabel="Clusterzahl", ylim=limits)
    axis.set_xticks(cluster_metrics["clusters"])
    axis.grid(alpha=0.25)

axes[0].legend()
fig.suptitle("K-Means-Modellvergleich auf relativen Werteprofilen")
fig.tight_layout()
fig.savefig(
    FIGURE_DIRECTORY / "role_clustering_metrics.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

`k=4` bietet für diesen Snapshot den besten Kompromiss: höchste beziehungsweise praktisch höchste Silhouette, sehr hohe Startstabilität, bessere Davies-Bouldin-Trennung als die gröberen Lösungen und keine sehr kleinen Cluster. Die niedrige absolute Silhouette um 0,20 zeigt jedoch, dass keine scharf getrennten natürlichen Klassen vorliegen.

## Interpretation von k=4

In [ ]:
cluster_model = KMeans(
    n_clusters=SELECTED_CLUSTER_COUNT,
    random_state=RANDOM_STATE,
    n_init=20,
)
cluster_labels = cluster_model.fit_predict(scaled_stats)
cluster_distances = cluster_model.transform(scaled_stats)

cluster_names = {
    cluster: f"C{cluster + 1}"
    for cluster in range(SELECTED_CLUSTER_COUNT)
}
clustered = data.copy()
clustered["cluster"] = cluster_labels
clustered["distance_to_centroid"] = cluster_distances[
    np.arange(len(data)), cluster_labels
]
cluster_sizes = clustered.groupby("cluster").size().reindex(cluster_names)

final_flags = data["is_final_evolution"].astype(str).str.lower().eq("true")
profile_table = (
    relative_stats.assign(cluster=cluster_labels)
    .groupby("cluster")[STAT_COLUMNS]
    .mean()
    .mul(100)
)
profile_table.insert(0, "pokemon_count", cluster_sizes)
profile_table["average_bst"] = clustered.groupby("cluster")[
    "base_stat_total"
].mean()
profile_table["average_evolution_stage"] = clustered.groupby("cluster")[
    "evolution_stage"
].mean()
profile_table["final_evolution_pct"] = (
    pd.DataFrame({"cluster": cluster_labels, "is_final": final_flags})
    .groupby("cluster")["is_final"]
    .mean()
    .mul(100)
)
profile_table = profile_table.rename(index=cluster_names)
display(profile_table.round(2))

In [ ]:
cluster_profile_percent = profile_table[STAT_COLUMNS]
overall_profile_percent = relative_stats[STAT_COLUMNS].mean().mul(100)
profile_index = cluster_profile_percent.div(overall_profile_percent).mul(100)

fig, axis = plt.subplots(figsize=(10, 4.5))
image = axis.imshow(profile_index, cmap="coolwarm", vmin=60, vmax=160)
axis.set_xticks(range(len(STAT_COLUMNS)), labels=STAT_COLUMNS, rotation=30, ha="right")
axis.set_yticks(range(len(profile_index.index)), labels=profile_index.index)
axis.set_title("Clusterprofile relativ zum Datensatzmittel (= 100)")

for row in range(profile_index.shape[0]):
    for column in range(profile_index.shape[1]):
        axis.text(
            column,
            row,
            f"{profile_index.iloc[row, column]:.0f}",
            ha="center",
            va="center",
        )

fig.colorbar(image, ax=axis, label="Index")
fig.tight_layout()
fig.savefig(
    FIGURE_DIRECTORY / "role_cluster_profiles.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

## Überlappung mit den transparenten Rollen

Die Rollen wurden nicht zum Fitten verwendet. Ihre Anteile helfen ausschließlich bei der nachträglichen fachlichen Interpretation. Da das Rollenmodell Multi-Label-Ausgaben liefert, können sich die Prozentwerte einer Clusterzeile auf mehr als 100 Prozent summieren.

In [ ]:
cluster_by_id = dict(
    zip(data["id"].astype(int), cluster_labels, strict=True)
)
role_rows = [
    {
        "cluster": cluster_by_id[pokemon_id],
        "role": score["role"].value,
    }
    for pokemon_id, scores in role_analysis.items()
    for score in scores
    if score["assigned"]
]
role_frame = pd.DataFrame(role_rows)
role_counts = pd.crosstab(role_frame["cluster"], role_frame["role"]).reindex(
    index=range(SELECTED_CLUSTER_COUNT),
    columns=[role.value for role in Role],
    fill_value=0,
)
role_percentages = (
    role_counts.div(cluster_sizes, axis=0)
    .mul(100)
    .rename(index=cluster_names)
)
display(role_percentages.round(1))

## Repräsentative Pokémon

Die folgenden Pokémon besitzen innerhalb ihres Clusters die geringsten standardisierten Distanzen zum jeweiligen Zentrum.

In [ ]:
representative_rows = []

for cluster in range(SELECTED_CLUSTER_COUNT):
    cluster_indices = np.flatnonzero(cluster_labels == cluster)
    closest_indices = cluster_indices[
        np.argsort(cluster_distances[cluster_indices, cluster])[:10]
    ]
    for rank, pokemon_index in enumerate(closest_indices, start=1):
        representative_rows.append(
            {
                "cluster": cluster_names[cluster],
                "rank": rank,
                "pokemon": data.iloc[pokemon_index]["name"],
                "distance": cluster_distances[pokemon_index, cluster],
            }
        )

representatives = pd.DataFrame(representative_rows)
display(
    representatives.pivot(index="rank", columns="cluster", values="pokemon")
)

## Schlussfolgerung

Das Clustering bestätigt vier breite Wertearchetypen: physisch-schnell, physisch-defensiv, speziell offensiv und HP-/spezialdefensiv. Ähnliche durchschnittliche Gesamtbasiswerte, Evolutionsstufen und Finalanteile zeigen, dass die relative Darstellung nicht lediglich Stärke oder Entwicklungsfortschritt rekonstruiert.

Die absolute Silhouette bleibt jedoch niedrig, und mehrere transparente Rollen überlappen innerhalb jedes Clusters. K-Means erzwingt außerdem genau ein hartes Label, während Pokémon wie Alakazam oder Garchomp nachvollziehbar mehrere Rollen erfüllen. Das Clustering wird deshalb als explorative Validierung dokumentiert, aber nicht als Produktionssignal in den Recommender übernommen. Die regelbasierte Multi-Label-Baseline bleibt erklärbarer, granularer und fachlich besser kontrollierbar.